# Pull data from SSMS
This notebook pulls existing data from sql SSMS and exports it to folder data/01_sql_input

In [7]:
from pathlib import Path
from collections.abc import Mapping
import json
import warnings

import numpy as np
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

server = "localhost"
database = "RevenueAnalytics"

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

OUTPUT_DIR_SQL_INPUT = Path("../data/sql_input")
OUTPUT_DIR_SQL_INPUT_SAMPLE = Path("../data/01_sql_input")
OUTPUT_DIR_SQL_INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_SQL_INPUT_SAMPLE.mkdir(parents=True, exist_ok=True)
print(f"Output folder preprocessed: {OUTPUT_DIR_SQL_INPUT.resolve()}")
print(f"Output folder preprocessed: {OUTPUT_DIR_SQL_INPUT_SAMPLE.resolve()}")

Output folder preprocessed: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\sql_input
Output folder preprocessed: C:\Users\Anast\OneDrive\Desktop\AS Portfolio\early-warning\data\01_sql_input


In [2]:
# Connection ssms
connection_url = URL.create(
    "mssql+pyodbc",
    host=server,
    database=database,
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "trusted_connection": "yes",
        "TrustServerCertificate": "yes"})

engine = create_engine(connection_url)

# Verify connection
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT @@SERVERNAME AS server_name, DB_NAME() AS database_name"))
    
    print(result.fetchone())

C:\Users\Anast\AppData\Local\Temp\ipykernel_25956\1869408425.py:14: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as connection:


('Ana', 'RevenueAnalytics')


In [ ]:
# Load first rows from sales table
query = """
SELECT TOP 100 *
FROM raw.factSales
"""

with engine.connect() as connection:
    df = pd.read_sql(text(query), connection)

df.head()

,sales_id,order_id,date_id,date,year_month,customer_id,product_id,region_id,sales_rep_id,units,asp_eur,discount_pct,revenue_eur,revenue_local_currency,currency,unit_cost_eur,gross_profit_eur,gross_margin_pct,is_outlier_order
0,1,ORD-00000001,20220304,2022-03-04,2022-03-01 00:00:00.0000000,282,1133,2,39,26,104.00,0.092,2704.13,2704.13,EUR,78.47,663.79,0.2455,False
1,2,ORD-00000002,20240503,2024-05-03,2024-05-01 00:00:00.0000000,562,1105,2,73,21,1698.07,0.089,35659.45,35659.45,EUR,944.57,15823.42,0.4437,False
2,3,ORD-00000003,20241205,2024-12-05,2024-12-01 00:00:00.0000000,146,1007,7,41,117,983.15,0.123,115029.00,169160.29,CAD,637.92,40392.64,0.3512,False
3,4,ORD-00000004,20241119,2024-11-19,2024-11-01 00:00:00.0000000,487,1176,6,10,5,63.48,0.084,317.39,344.99,USD,55.67,39.04,0.1230,False
4,5,ORD-00000005,20220713,2022-07-13,2022-07-01 00:00:00.0000000,869,1104,10,81,61,931.87,0.157,56844.25,9168427.42,JPY,728.64,12397.02,0.2181,False


In [9]:
TABLES: Mapping[str, tuple[str, str]] = {
    "sales": ("staging", "factSales"),
    "products": ("staging", "dimProduct"),
    "customers": ("staging", "dimCustomer"),
    "sales_reps": ("staging", "dimSalesRep"),
    "costs": ("staging", "factCosts"),
    "returns": ("staging", "factReturns"),
    "regions": ("staging", "dimRegion"),
    "inventory": ("staging", "factInventory"),
    "crm_activities": ("staging", "factCRMActivities"),
    "pipeline": ("staging", "factPipeline"),
    "dates": ("curated", "dimDate")}

dataframes = {
    name: pd.read_sql_table(
        table_name=table_name,
        schema=schema,
        con=engine)
    for name, (schema, table_name) in TABLES.items()}

sales = dataframes["sales"]
products = dataframes["products"]
customers = dataframes["customers"]
sales_reps = dataframes["sales_reps"]
costs = dataframes["costs"]
returns = dataframes["returns"]
regions = dataframes["regions"]
inventory = dataframes["inventory"]
crm_activities = dataframes["crm_activities"]
pipeline = dataframes["pipeline"]
dates = dataframes["dates"]

In [ ]:
# Adjustments
products["launch_date"] = pd.to_datetime(products["launch_year"].astype(str) + "-01-01", errors="coerce")

In [11]:
# validation: run uniqueness checks
checks = {
    "duplicate_product_ids": int(products["product_id"].duplicated().sum()),
    "duplicate_customer_ids": int(customers["customer_id"].duplicated().sum()),
    "duplicate_transaction_ids": int(sales["sales_id"].duplicated().sum()),
    "sales_unknown_products": int((~sales["product_id"].isin(products["product_id"])).sum()),
    "sales_unknown_customers": int((~sales["customer_id"].isin(customers["customer_id"])).sum())}

checks

{'duplicate_product_ids': 0,
 'duplicate_customer_ids': 0,
 'duplicate_transaction_ids': 0,
 'sales_unknown_products': 0,
 'sales_unknown_customers': 0}

In [12]:
# Map output filenames back to DataFrames
tables = {
    "sales": sales,
    "products": products,
    "customers": customers,
    "sales_reps": sales_reps,
    "costs": costs,
    "returns": returns,
    "regions": regions,
    "inventory": inventory,
    "crm_activities": crm_activities,
    "pipeline": pipeline,
    "dates": dates}

## Export to csv

In [13]:
# Export full tables and samples of up to 100 rows
for table_name, df in tables.items():
    full_file = OUTPUT_DIR_SQL_INPUT / f"{table_name}.csv"
    sample_file = OUTPUT_DIR_SQL_INPUT_SAMPLE / f"{table_name}_sample.csv"

    # Full dataset
    df.to_csv(
        full_file,
        index=False,
        encoding="utf-8")

    # Reproducible random sample, or all rows when fewer than 100 exist
    sample_df = df.sample(
        n=min(100, len(df)),
        random_state=42)

    sample_df.to_csv(
        sample_file,
        index=False,
        encoding="utf-8")

    print(
        f"Exported {table_name}: "
        f"{len(df):,} full rows and {len(sample_df):,} sample rows")

Exported sales: 120,000 full rows and 100 sample rows
Exported products: 200 full rows and 100 sample rows
Exported customers: 1,200 full rows and 100 sample rows
Exported sales_reps: 85 full rows and 85 sample rows
Exported costs: 12,000 full rows and 100 sample rows
Exported returns: 4,200 full rows and 100 sample rows
Exported regions: 10 full rows and 10 sample rows
Exported inventory: 69,852 full rows and 100 sample rows
Exported crm_activities: 35,000 full rows and 100 sample rows
Exported pipeline: 15,000 full rows and 100 sample rows
Exported dates: 4,018 full rows and 100 sample rows


In [14]:
engine.dispose()